# `Dynamic AI Knowledge Assistant`

In [45]:
# ============================================================
# STEP 1  - Inspect Phase 5 Artifacts
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import time

PROJECT_ROOT = Path(r"C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHUNK_METADATA_DIR = PROCESSED_DIR / "chunk_metadata"
EMBEDDINGS_DIR = PROCESSED_DIR / "embeddings"
CHUNK_METADATA_PATH = ( CHUNK_METADATA_DIR / "chunk_metadata.csv")

MINILM_EMBEDDINGS_PATH = (
    EMBEDDINGS_DIR / "all_MiniLM_L6_v2_embeddings.npy"
)
MPNET_EMBEDDINGS_PATH = (
    EMBEDDINGS_DIR / "all_mpnet_base_v2_embeddings.npy"
)

print("=" * 70)
print("ARTIFACT INSPECTION")
print("=" * 70)

print("\nChunk metadata:", CHUNK_METADATA_PATH)
print("Exists:", CHUNK_METADATA_PATH.exists())

print("\nMiniLM embeddings:", MINILM_EMBEDDINGS_PATH)
print("Exists:", MINILM_EMBEDDINGS_PATH.exists())

print("\nMPNet embeddings:" , MPNET_EMBEDDINGS_PATH)
print("Exists:", MPNET_EMBEDDINGS_PATH.exists())

ARTIFACT INSPECTION

Chunk metadata: C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\chunk_metadata\chunk_metadata.csv
Exists: True

MiniLM embeddings: C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\embeddings\all_MiniLM_L6_v2_embeddings.npy
Exists: True

MPNet embeddings: C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\embeddings\all_mpnet_base_v2_embeddings.npy
Exists: True


In [46]:
# ============================================================
# STEP 2 -  Load Chunk Metadata
# ============================================================

chunk_metadata_df = pd.read_csv(CHUNK_METADATA_PATH)

print("Chunk metadata shape:", chunk_metadata_df.shape)
print("\nColumns:", chunk_metadata_df.columns.tolist())
display("First 5 records:", chunk_metadata_df.head())


Chunk metadata shape: (4693, 9)

Columns: ['chunk_id', 'article_id', 'title', 'category', 'source_url', 'original_file_name', 'chunk_index', 'chunk_text', 'chunk_length']


'First 5 records:'

,chunk_id,article_id,title,category,source_url,original_file_name,chunk_index,chunk_text,chunk_length
0,ART_84789a287028_chunk_0000,ART_84789a287028,1.58-bit large language model,Large Language Model,https://en.wikipedia.org/wiki/1.58-bit_large_l...,1.58-bit_large_language_model.txt,0,A 1.58-bit large language model (also known as...,809
1,ART_84789a287028_chunk_0001,ART_84789a287028,1.58-bit large language model,Large Language Model,https://en.wikipedia.org/wiki/1.58-bit_large_l...,1.58-bit_large_language_model.txt,1,"The name ""1.58-bit"" comes from the fact that a...",319
2,ART_84789a287028_chunk_0002,ART_84789a287028,1.58-bit large language model,Large Language Model,https://en.wikipedia.org/wiki/1.58-bit_large_l...,1.58-bit_large_language_model.txt,2,"== BitNet == In 2024, Ma et al., researchers a...",600
3,ART_84789a287028_chunk_0003,ART_84789a287028,1.58-bit large language model,Large Language Model,https://en.wikipedia.org/wiki/1.58-bit_large_l...,1.58-bit_large_language_model.txt,3,== Post-training quantization == BitNet derive...,438
4,ART_84789a287028_chunk_0004,ART_84789a287028,1.58-bit large language model,Large Language Model,https://en.wikipedia.org/wiki/1.58-bit_large_l...,1.58-bit_large_language_model.txt,4,== Critique == Some researchers point out that...,245


In [47]:
# ============================================================
# STEP 3 - Load MiniLM Embeddings
# ============================================================

embeddings = np.load(MINILM_EMBEDDINGS_PATH)

print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)
print("Embedding dimensions:", embeddings.shape[1])
print("Number of embeddings:", embeddings.shape[0])


Embedding shape: (4693, 384)
Embedding dtype: float32
Embedding dimensions: 384
Number of embeddings: 4693


In [48]:
# ============================================================
# STEP 4 - Validate Metadata / Embedding Alignment
# ============================================================

print("=" * 70)
print("METADATA / EMBEDDING ALIGNMENT VALIDATION")
print("=" * 70)

metadata_count = len(chunk_metadata_df)
embedding_count = embeddings.shape[0]
embedding_dimension = embeddings.shape[1]

print("Metadata records :", metadata_count)
print("Embedding vectors:", embedding_count)
print("Embedding dimension:", embedding_dimension)

assert metadata_count == 4693, (
    f"Expected 4693 metadata records, "
    f"found {metadata_count}"
)

assert embedding_count == 4693, (
    f"Expected 4693 embeddings, "
    f"found {embedding_count}"
)

assert metadata_count == embedding_count, (
    "Metadata and embedding counts do not match"
)

assert embedding_dimension == 384, (
    f"Expected 384 dimensions, "
    f"found {embedding_dimension}"
)

assert np.isfinite(embeddings).all(), (
    "Embeddings contain NaN or Inf values"
)

print("\nMetadata count          : PASSED")
print("Embedding count         : PASSED")
print("Metadata/embedding match: PASSED")
print("Embedding dimensions    : PASSED")
print("Finite values           : PASSED")

# ============================================================
# Validate Chunk IDs
# ============================================================

assert "chunk_id" in chunk_metadata_df.columns

assert (
    chunk_metadata_df["chunk_id"].notna().all()
)

assert (
    chunk_metadata_df["chunk_id"].nunique()
    == len(chunk_metadata_df)
)

print("Chunk IDs present        : PASSED")
print("Chunk IDs unique         : PASSED")

# ============================================================
# Validate Embedding Normalization
# ============================================================

embedding_norms = np.linalg.norm(
    embeddings,
    axis=1
)

print("Minimum embedding norm:",
      embedding_norms.min())

print("Maximum embedding norm:",
      embedding_norms.max())

assert np.allclose(
    embedding_norms,
    1.0,
    atol=1e-5
), "Embeddings are not normalized."

print("Embedding normalization: PASSED")

METADATA / EMBEDDING ALIGNMENT VALIDATION
Metadata records : 4693
Embedding vectors: 4693
Embedding dimension: 384

Metadata count          : PASSED
Embedding count         : PASSED
Metadata/embedding match: PASSED
Embedding dimensions    : PASSED
Finite values           : PASSED
Chunk IDs present        : PASSED
Chunk IDs unique         : PASSED
Minimum embedding norm: 0.9999999
Maximum embedding norm: 1.0000001
Embedding normalization: PASSED


In [49]:
# ============================================================
# STEP 5 -  Build FAISS Index
# ============================================================

import faiss

embedding_dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(embeddings.astype("float32"))

print("FAISS index created successfully.")
print("Index dimension:",faiss_index.d)
print("Vectors stored:", faiss_index.ntotal)


# ============================================================
# FAISS Index Validation
# ============================================================

assert faiss_index.d == 384
assert faiss_index.ntotal == 4693

# ============================================================
# VECTOR DATABASE DIRECTORY
# ============================================================

VECTOR_DB_DIR = (PROCESSED_DIR / "vector_db")

FAISS_DIR = (VECTOR_DB_DIR / "faiss")
CHROMA_DIR = (VECTOR_DB_DIR / "chroma")
FAISS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True,exist_ok=True)
print("\nVector database directories created:")
print(FAISS_DIR)
print(CHROMA_DIR)


# ============================================================
# Save FAISS Index
# ============================================================

FAISS_INDEX_PATH = (FAISS_DIR / "minilm_faiss.index")
faiss.write_index(faiss_index, str(FAISS_INDEX_PATH))
print("\nFAISS index saved:")
print(FAISS_INDEX_PATH)

FAISS index created successfully.
Index dimension: 384
Vectors stored: 4693

Vector database directories created:
C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\vector_db\faiss
C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\vector_db\chroma

FAISS index saved:
C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\vector_db\faiss\minilm_faiss.index


In [50]:
test_query = ( "What is artificial intelligence?")

# ============================================================
# FAISS Retrieval Test
# ============================================================

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

query_embedding = embedding_model.encode(
    [test_query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_embedding = query_embedding.astype(
    "float32"
)

print("Query embedding shape:")
print(query_embedding.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16266.17it/s]


Query embedding shape:
(1, 384)


In [51]:
# ============================================================
# FAISS Similarity Search
# ============================================================

TOP_K = 5

scores, indices = faiss_index.search(
    query_embedding,
    TOP_K
)

print("Retrieved indices:")
print(indices)

print("\nSimilarity scores:")
print(scores)

# ============================================================
# Display Retrieved Chunks
# ============================================================

results = chunk_metadata_df.iloc[
    indices[0]
].copy()

results["similarity_score"] = scores[0]

display(
    results[
        [
            "chunk_id",
            "article_id",
            "title",
            "chunk_index",
            "chunk_text",
            "similarity_score"
        ]
    ]
)

Retrieved indices:
[[ 436 1730 1712  788  603]]

Similarity scores:
[[0.78055114 0.7772615  0.73477346 0.7202707  0.71426636]]


,chunk_id,article_id,title,chunk_index,chunk_text,similarity_score
436,ART_ede7794b399f_chunk_0000,ART_ede7794b399f,Artificial intelligence,0,Artificial intelligence (AI) is the capability...,0.780551
1730,ART_7b50a3a1d378_chunk_0018,ART_7b50a3a1d378,Glossary of artificial intelligence,18,The algorithms are typically modeled after the...,0.777261
1712,ART_7b50a3a1d378_chunk_0000,ART_7b50a3a1d378,Glossary of artificial intelligence,0,This glossary of artificial intelligence is a ...,0.734773
788,ART_5b75b0467a43_chunk_0046,ART_5b75b0467a43,Computer science,46,==== Artificial intelligence ==== Artificial i...,0.720271
603,ART_ede7794b399f_chunk_0167,ART_ede7794b399f,Artificial intelligence,167,=== Other sources === == External links == Hau...,0.714266


In [52]:
# ============================================================
# STEP 6 -  Create Persistent ChromaDB Client
# ============================================================

import chromadb

CHROMA_PATH = str(CHROMA_DIR)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
print("ChromaDB persistent client created.")
print("Storage location:", CHROMA_PATH)


ChromaDB persistent client created.
Storage location: C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\vector_db\chroma


In [53]:
# ============================================================
# STEP 7 -  Create ChromaDB Collection
# ============================================================

COLLECTION_NAME = "wiki_chunks_minilm"

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={
        "description": "Wikipedia semantic chunks embedded using all-MiniLM-L6-v2",
        "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
        "embedding_dimension": 384,
        "distance_metric": "cosine"
    },
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
)

print("Collection created:", COLLECTION_NAME)
print("Current collection count:", collection.count())

# ============================================================
# Inspect Metadata Schema
# ============================================================

print("\nColumns: ", chunk_metadata_df.columns.tolist())

faiss_scores, faiss_indices = faiss_index.search(
    query_embedding,
    TOP_K
)


Collection created: wiki_chunks_minilm
Current collection count: 4693

Columns:  ['chunk_id', 'article_id', 'title', 'category', 'source_url', 'original_file_name', 'chunk_index', 'chunk_text', 'chunk_length']


In [54]:
# ============================================================
# STEP 8 -  Prepare ChromaDB Metadata
# ============================================================

chroma_metadatas = []

for _, row in chunk_metadata_df.iterrows():

    metadata = {
        "article_id": str(row["article_id"]),
        "title": str(row["title"]),
        "category": str(row["category"]),
        "source_url": str(row["source_url"]),
        "original_file_name": str(row["original_file_name"]),
        "chunk_index": int(row["chunk_index"]),
        "chunk_length": int(row["chunk_length"])
    }

    chroma_metadatas.append(metadata)

print("ChromaDB metadata records prepared:")
print(len(chroma_metadatas))

# ============================================================
# ChromaDB Metadata Validation
# ============================================================

assert len(chroma_metadatas) == 4693

for metadata in chroma_metadatas:

    assert metadata["article_id"]
    assert metadata["title"]
    assert metadata["category"]
    assert metadata["source_url"]
    assert metadata["original_file_name"]
    assert metadata["chunk_index"] >= 0
    assert metadata["chunk_length"] > 0

print("ChromaDB metadata count : PASSED")
print("Article traceability    : PASSED")
print("Source URL preservation : PASSED")
print("Chunk index validation  : PASSED")
print("Chunk length validation : PASSED")

ChromaDB metadata records prepared:
4693
ChromaDB metadata count : PASSED
Article traceability    : PASSED
Source URL preservation : PASSED
Chunk index validation  : PASSED
Chunk length validation : PASSED


In [55]:
# ============================================================
# STEP 9 - Prepare ChromaDB Records
# ============================================================

ids = (
    chunk_metadata_df["chunk_id"]
    .astype(str)
    .tolist()
)

documents = (
    chunk_metadata_df["chunk_text"]
    .astype(str)
    .tolist()
)

embedding_list = (
    embeddings
    .astype("float32")
    .tolist()
)

print("IDs:", len(ids))
print("Documents:", len(documents))
print("Embeddings:", len(embedding_list))
print("Metadata:", len(chroma_metadatas))

assert len(ids) == 4693
assert len(documents) == 4693
assert len(embedding_list) == 4693
assert len(chroma_metadatas) == 4693

assert len(ids) == len(documents)
assert len(ids) == len(embedding_list)
assert len(ids) == len(chroma_metadatas)

assert len(set(ids)) == 4693

print("ChromaDB record alignment: PASSED")

IDs: 4693
Documents: 4693
Embeddings: 4693
Metadata: 4693
ChromaDB record alignment: PASSED


In [56]:
# ============================================================
# STEP 10 - Insert Embeddings + Documents + Metadata
#           into ChromaDB
# ============================================================

BATCH_SIZE = 500

total_records = len(ids)

print("=" * 70)
print("CHROMADB INSERTION")
print("=" * 70)

print("Total records:", total_records)
print("Batch size:", BATCH_SIZE)

for start in range(0, total_records, BATCH_SIZE):

    end = min(
        start + BATCH_SIZE,
        total_records
    )

    collection.upsert(
        ids=ids[start:end],
        embeddings=embedding_list[start:end],
        documents=documents[start:end],
        metadatas=chroma_metadatas[start:end]
    )

    print(
        f"Upserted records {start} - {end - 1}"
    )

print("\nChromaDB insertion/upsert completed.")
print("Collection count:", collection.count())

CHROMADB INSERTION
Total records: 4693
Batch size: 500
Upserted records 0 - 499
Upserted records 500 - 999
Upserted records 1000 - 1499
Upserted records 1500 - 1999
Upserted records 2000 - 2499
Upserted records 2500 - 2999
Upserted records 3000 - 3499
Upserted records 3500 - 3999
Upserted records 4000 - 4499
Upserted records 4500 - 4692

ChromaDB insertion/upsert completed.
Collection count: 4693


In [57]:
# ============================================================
# STEP 11 -  ChromaDB Collection Validation
# ============================================================

chroma_count = collection.count()

print("=" * 70)
print("CHROMADB VALIDATION")
print("=" * 70)

print("Expected records :", 4693)
print("Actual records   :", chroma_count)
assert chroma_count == 4693, (f"Expected 4693 records, found {chroma_count}")
print("\nChromaDB record count: PASSED")

# ============================================================
# Verify Stored ChromaDB Records
# ============================================================

sample = collection.get(
    limit=5,
    include=[
        "documents",
        "metadatas",
        "embeddings"
    ]
)

print("Retrieved sample records:", len(sample["ids"]))

print("\nFirst 5 IDs:")
for chunk_id in sample["ids"]:
    print(chunk_id)

print("\nFirst record metadata:")
print(sample["metadatas"][0])

print("\nFirst record text:")
print(sample["documents"][0][:500])

CHROMADB VALIDATION
Expected records : 4693
Actual records   : 4693

ChromaDB record count: PASSED
Retrieved sample records: 5

First 5 IDs:
ART_84789a287028_chunk_0000
ART_84789a287028_chunk_0001
ART_84789a287028_chunk_0002
ART_84789a287028_chunk_0003
ART_84789a287028_chunk_0004

First record metadata:
{'title': '1.58-bit large language model', 'chunk_index': 0, 'original_file_name': '1.58-bit_large_language_model.txt', 'chunk_length': 809, 'article_id': 'ART_84789a287028', 'source_url': 'https://en.wikipedia.org/wiki/1.58-bit_large_language_model', 'category': 'Large Language Model'}

First record text:
A 1.58-bit large language model (also known as a ternary LLM) is a type of large language model (LLM) designed to be computationally efficient. It achieves this by using weights that are restricted to only three values: −1, 0, and +1. This restriction significantly reduces the model's memory footprint and allows for faster processing, as computationally expensive multiplication operat

In [58]:
test_query = "What is artificial intelligence?"

# ============================================================
# STEP 12 -  ChromaDB Semantic Retrieval Test
# ============================================================

TOP_K = 5

chroma_results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=TOP_K,
    include=[
        "documents",
        "metadatas",
        "distances"
    ]
)

print("=" * 70)
print("CHROMADB RETRIEVAL TEST")
print("=" * 70)

print("Query:", test_query)
print("Number of results:", len(chroma_results["ids"][0]))

# ============================================================
# Display ChromaDB Retrieval Results
# ============================================================

for rank in range(TOP_K):

    print("\n" + "=" * 80)
    print(f"RANK {rank + 1}")
    print("=" * 80)

    print(
        "Chunk ID:",
        chroma_results["ids"][0][rank]
    )

    metadata = chroma_results["metadatas"][0][rank]

    print(
        "Article ID:",
        metadata["article_id"]
    )

    print(
        "Title:",
        metadata["title"]
    )

    print(
        "Category:",
        metadata["category"]
    )

    print(
        "Chunk Index:",
        metadata["chunk_index"]
    )

    print(
        "Source URL:",
        metadata["source_url"]
    )

    print(
        "Distance:",
        chroma_results["distances"][0][rank]
    )

    print("\nChunk text:")
    print(
        chroma_results["documents"][0][rank][:500]
    )


CHROMADB RETRIEVAL TEST
Query: What is artificial intelligence?
Number of results: 5

RANK 1
Chunk ID: ART_ede7794b399f_chunk_0000
Article ID: ART_ede7794b399f
Title: Artificial intelligence
Category: Artificial Intelligence
Chunk Index: 0
Source URL: https://en.wikipedia.org/wiki/Artificial_intelligence
Distance: 0.21944868564605713

Chunk text:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals. High-p

RANK 2
Chunk ID: ART_7b50a3a1d378_chunk_0018
Article ID: ART_7b50a3a1d378
Title: Glossary of artificial intelligence
Category: Artificial Intelligenc

In [ ]:
# ===================================================================
# STEP 12 - Convert ChromaDB Cosine Distance to Cosine Similarity
# ===================================================================

chroma_distances = np.array(
    chroma_results["distances"][0]
)

chroma_similarities = (
    1 - chroma_distances
)

print("ChromaDB cosine similarities:")

for rank, similarity in enumerate(
    chroma_similarities,
    start=1
):
    print(
        f"Rank {rank}: "
        f"{similarity:.6f}"
    )

ChromaDB cosine similarities:
Rank 1: 0.780551
Rank 2: 0.777261
Rank 3: 0.734774
Rank 4: 0.720271
Rank 5: 0.714266


In [62]:
# ============================================================
# STEP 13 - FAISS vs ChromaDB Retrieval Comparison
# ============================================================

# ------------------------------------------------------------
# 1. Validate actual FAISS search output
# ------------------------------------------------------------

assert faiss_scores.shape == (1, TOP_K)
assert faiss_indices.shape == (1, TOP_K)

print("FAISS retrieval output : PASSED")


# ------------------------------------------------------------
# 2. Extract actual FAISS results
# ------------------------------------------------------------

faiss_similarity = faiss_scores[0]
faiss_index_values = faiss_indices[0]

assert len(faiss_similarity) == TOP_K
assert len(faiss_index_values) == TOP_K


# ------------------------------------------------------------
# 3. Get FAISS chunk IDs
# ------------------------------------------------------------

faiss_chunk_ids = (
    chunk_metadata_df.iloc[faiss_index_values]["chunk_id"]
    .astype(str)
    .tolist()
)


# ------------------------------------------------------------
# 4. Get actual ChromaDB results
# ------------------------------------------------------------

chroma_chunk_ids = chroma_results["ids"][0]

chroma_distances = np.array(
    chroma_results["distances"][0]
)

chroma_similarity = 1 - chroma_distances


# ------------------------------------------------------------
# 5. Validate ChromaDB results
# ------------------------------------------------------------

assert len(chroma_similarity) == TOP_K
assert len(chroma_chunk_ids) == TOP_K


# ------------------------------------------------------------
# 6. Compare similarity scores
# ------------------------------------------------------------

score_difference = np.abs(
    faiss_similarity - chroma_similarity
)


# ------------------------------------------------------------
# 7. Comparison table
# ------------------------------------------------------------

comparison_df = pd.DataFrame({
    "Rank": range(1, TOP_K + 1),
    "FAISS_Chunk_ID": faiss_chunk_ids,
    "ChromaDB_Chunk_ID": chroma_chunk_ids,
    "FAISS_Similarity": faiss_similarity,
    "ChromaDB_Similarity": chroma_similarity,
    "Absolute_Difference": score_difference
})

display(comparison_df)


# ------------------------------------------------------------
# 8. Final comparison validation
# ------------------------------------------------------------

print("=" * 70)
print("FAISS vs ChromaDB COMPARISON")
print("=" * 70)

print(
    "Maximum score difference:",
    score_difference.max()
)

print(
    "Mean score difference:",
    score_difference.mean()
)

assert faiss_chunk_ids == chroma_chunk_ids

assert np.allclose(
    faiss_similarity,
    chroma_similarity,
    atol=1e-5
)

print("\nRetrieved chunk IDs : PASSED")
print("Similarity scores   : PASSED")
print("FAISS/ChromaDB      : CONSISTENT")

FAISS retrieval output : PASSED


,Rank,FAISS_Chunk_ID,ChromaDB_Chunk_ID,FAISS_Similarity,ChromaDB_Similarity,Absolute_Difference
0,1,ART_ede7794b399f_chunk_0000,ART_ede7794b399f_chunk_0000,0.780551,0.780551,1.788139e-07
1,2,ART_7b50a3a1d378_chunk_0018,ART_7b50a3a1d378_chunk_0018,0.777261,0.777261,0.000000e+00
2,3,ART_7b50a3a1d378_chunk_0000,ART_7b50a3a1d378_chunk_0000,0.734773,0.734774,1.788139e-07
3,4,ART_5b75b0467a43_chunk_0046,ART_5b75b0467a43_chunk_0046,0.720271,0.720271,5.960464e-08
4,5,ART_ede7794b399f_chunk_0167,ART_ede7794b399f_chunk_0167,0.714266,0.714266,5.960464e-08


FAISS vs ChromaDB COMPARISON
Maximum score difference: 1.7881393432617188e-07
Mean score difference: 9.5367431640625e-08

Retrieved chunk IDs : PASSED
Similarity scores   : PASSED
FAISS/ChromaDB      : CONSISTENT


In [66]:
# ============================================================
# STEP 14 - FINAL VALIDATION
# ============================================================

print("=" * 70)
print("PHASE 6 – VECTOR DATABASE FINAL VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Embedding validation
# ------------------------------------------------------------

assert embeddings.shape == (4693, 384)

print("Embedding count        : PASSED")
print("Embedding dimensions   : PASSED")


# ------------------------------------------------------------
# 2. FAISS validation
# ------------------------------------------------------------

assert faiss_index.ntotal == 4693

print("FAISS vector count     : PASSED")


# ------------------------------------------------------------
# 3. ChromaDB validation
# ------------------------------------------------------------

assert collection.count() == 4693

print("ChromaDB vector count  : PASSED")


# ------------------------------------------------------------
# 4. Metadata validation
# ------------------------------------------------------------

assert len(chunk_metadata_df) == 4693

print("Chunk metadata count   : PASSED")


# ------------------------------------------------------------
# 5. Chunk IDs
# ------------------------------------------------------------

assert (
    chunk_metadata_df["chunk_id"]
    .nunique()
    == 4693
)

print("Unique chunk IDs       : PASSED")


# ------------------------------------------------------------
# 6. Article coverage
# ------------------------------------------------------------

assert (
    chunk_metadata_df["article_id"]
    .nunique()
    == 128
)

print("Article coverage       : PASSED")


# ------------------------------------------------------------
# 7. Retrieval validation
# ------------------------------------------------------------

assert len(faiss_similarity) == TOP_K
assert len(chroma_similarity) == TOP_K

print("FAISS retrieval        : PASSED")
print("ChromaDB retrieval     : PASSED")


# ------------------------------------------------------------
# 8. Retrieval consistency
# ------------------------------------------------------------

assert np.allclose(
    faiss_similarity,
    chroma_similarity,
    atol=1e-5
)
print("Retrieval consistency  : PASSED")


print("\n" + "=" * 70)
print("VALIDATION STATUS: PASSED")
print("=" * 70)

PHASE 6 – VECTOR DATABASE FINAL VALIDATION
Embedding count        : PASSED
Embedding dimensions   : PASSED
FAISS vector count     : PASSED
ChromaDB vector count  : PASSED
Chunk metadata count   : PASSED
Unique chunk IDs       : PASSED
Article coverage       : PASSED
FAISS retrieval        : PASSED
ChromaDB retrieval     : PASSED
Retrieval consistency  : PASSED

VALIDATION STATUS: PASSED
